# ============================================
# MODULE 2: MACHINE LEARNING INTRODUCTION
# ============================================
#
# Learning Objectives:
# - Understand fundamental concepts of machine learning
# - Distinguish between supervised and unsupervised learning
# - Learn the machine learning workflow and best practices
# - Apply basic ML algorithms to power systems data
# - Understand when to use different types of algorithms
# - Implement train-test split and cross-validation
#
# Real-World Application:
# Machine learning transforms how electrical engineers solve problems:
# - Load forecasting: Predict future electricity demand (supervised regression)
# - Fault detection: Classify normal vs abnormal system states (supervised classification)
# - Equipment clustering: Group similar transformers for maintenance (unsupervised clustering)
# - Anomaly detection: Identify unusual patterns in sensor data (unsupervised)
# Understanding ML fundamentals enables you to choose the right algorithm for each problem.
#
# Estimated Time: 4-5 hours
# ============================================

## Section 1: Import Libraries and Setup

In [ ]:
# Import pandas for data manipulation
import pandas as pd

# Import numpy for numerical operations
import numpy as np

# Import matplotlib for basic plotting
import matplotlib.pyplot as plt

# Import seaborn for enhanced visualizations
import seaborn as sns

# Import datetime for time operations
from datetime import datetime, timedelta

# Import train_test_split for splitting data
# Essential for proper ML model evaluation
from sklearn.model_selection import train_test_split, cross_val_score, KFold

# Import preprocessing tools
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Import supervised learning algorithms
# Regression algorithms (predict continuous values)
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Classification algorithms (predict categories)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Import unsupervised learning algorithms
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA

# Import metrics for model evaluation
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Import warnings to suppress unnecessary messages
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 3)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"Scikit-learn available: Yes")

## Section 2: Understanding Machine Learning Fundamentals

### What is Machine Learning?

Machine Learning is the science of programming computers to learn from data without being explicitly programmed. Instead of writing rules, we let algorithms discover patterns.

### Types of Machine Learning:

1. **Supervised Learning**: Learn from labeled data (input-output pairs)
   - **Regression**: Predict continuous values (e.g., load forecasting)
   - **Classification**: Predict categories (e.g., fault type detection)

2. **Unsupervised Learning**: Find patterns in unlabeled data
   - **Clustering**: Group similar data points
   - **Dimensionality Reduction**: Reduce number of features
   - **Anomaly Detection**: Identify unusual patterns

3. **Reinforcement Learning**: Learn through trial and error (not covered in this course)

## Section 3: Generate Power Systems Dataset

In [ ]:
# Generate comprehensive power systems dataset for ML demonstration
np.random.seed(42)

# Generate 2000 hourly measurements
n_records = 2000

# Create timestamp range
date_range = pd.date_range(start='2023-01-01', periods=n_records, freq='H')

# Extract temporal features
hours = date_range.hour
day_of_week = date_range.dayofweek
day_of_year = date_range.dayofyear

# Generate load with realistic patterns
daily_pattern = 100 + 50 * np.sin((hours - 6) * np.pi / 12)
weekly_pattern = np.where(day_of_week < 5, 1.0, 0.85)
seasonal_pattern = 1.0 + 0.15 * np.cos((day_of_year - 15) * 2 * np.pi / 365)
load_mw = daily_pattern * weekly_pattern * seasonal_pattern + np.random.normal(0, 8, n_records)

# Generate temperature with seasonal variation
daily_temp_var = 5 * np.sin((hours - 14) * np.pi / 12)
seasonal_temp = 15 + 12 * np.sin((day_of_year - 80) * 2 * np.pi / 365)
temperature_c = seasonal_temp + daily_temp_var + np.random.normal(0, 2, n_records)

# Generate voltage with load-dependent variation
voltage_kv = 230 - (load_mw - load_mw.mean()) * 0.015 + np.random.normal(0, 1.5, n_records)

# Generate current based on power relationship
power_factor = np.random.uniform(0.88, 0.96, n_records)
current_a = (load_mw * 1000) / (np.sqrt(3) * voltage_kv * power_factor) + np.random.normal(0, 15, n_records)

# Generate frequency with small variations
frequency_hz = 60.0 + (load_mw - load_mw.mean()) * 0.0001 + np.random.normal(0, 0.015, n_records)

# Generate wind speed
wind_speed_ms = 8 + 4 * np.sin((day_of_year - 15) * 2 * np.pi / 365) + np.random.exponential(3, n_records)
wind_speed_ms = np.clip(wind_speed_ms, 0, 25)

# Generate solar irradiance (only during daylight)
solar_irradiance = np.where(
    (hours >= 6) & (hours <= 18),
    1000 * np.sin((hours - 6) * np.pi / 12) * (0.8 + 0.2 * np.random.random(n_records)),
    0
)

# Generate equipment status for classification examples
# Normal operation vs different fault types
# 0: Normal, 1: Voltage fault, 2: Frequency fault, 3: Overload
equipment_status = np.zeros(n_records, dtype=int)

# Introduce voltage faults (5% of data)
voltage_fault_idx = np.random.choice(n_records, size=int(0.05*n_records), replace=False)
equipment_status[voltage_fault_idx] = 1
voltage_kv[voltage_fault_idx] = voltage_kv[voltage_fault_idx] * np.random.uniform(0.85, 0.95, len(voltage_fault_idx))

# Introduce frequency faults (3% of data)
freq_fault_idx = np.random.choice([i for i in range(n_records) if i not in voltage_fault_idx], 
                                  size=int(0.03*n_records), replace=False)
equipment_status[freq_fault_idx] = 2
frequency_hz[freq_fault_idx] = 60.0 + np.random.uniform(-0.5, -0.3, len(freq_fault_idx))

# Introduce overload conditions (4% of data)
overload_idx = np.random.choice([i for i in range(n_records) if i not in voltage_fault_idx and i not in freq_fault_idx],
                               size=int(0.04*n_records), replace=False)
equipment_status[overload_idx] = 3
load_mw[overload_idx] = load_mw[overload_idx] * np.random.uniform(1.2, 1.4, len(overload_idx))

# Create DataFrame
df = pd.DataFrame({
    'timestamp': date_range,
    'hour': hours,
    'day_of_week': day_of_week,
    'is_weekend': (day_of_week >= 5).astype(int),
    'temperature_c': temperature_c,
    'load_mw': load_mw,
    'voltage_kv': voltage_kv,
    'current_a': current_a,
    'frequency_hz': frequency_hz,
    'power_factor': power_factor,
    'wind_speed_ms': wind_speed_ms,
    'solar_irradiance': solar_irradiance,
    'equipment_status': equipment_status
})

# Add cyclical features
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

print(f"Generated {len(df)} power system records")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"\nEquipment Status Distribution:")
status_labels = {0: 'Normal', 1: 'Voltage Fault', 2: 'Frequency Fault', 3: 'Overload'}
status_counts = df['equipment_status'].value_counts().sort_index()
for status, count in status_counts.items():
    print(f"  {status_labels[status]}: {count} ({count/len(df)*100:.1f}%)")

print("\nFirst 5 rows:")
print(df.head())

## Section 4: SUPERVISED LEARNING - REGRESSION

### Use Case: Load Forecasting
Predict future electricity load based on time, weather, and other factors.

In [ ]:
# Prepare data for regression (load forecasting)
# Goal: Predict load_mw from other features

# Select features (predictors)
feature_cols_regression = ['hour', 'day_of_week', 'is_weekend', 'temperature_c',
                          'voltage_kv', 'wind_speed_ms', 'solar_irradiance',
                          'hour_sin', 'hour_cos']

# Create feature matrix X and target vector y
X_reg = df[feature_cols_regression].copy()
y_reg = df['load_mw'].copy()

print("Regression Task: Load Forecasting")
print(f"Features (X): {X_reg.shape}")
print(f"Target (y): {y_reg.shape}")
print(f"\nFeature columns:\n{feature_cols_regression}")
print(f"\nTarget variable: load_mw")
print(f"Target range: {y_reg.min():.2f} to {y_reg.max():.2f} MW")

In [ ]:
# Split data into training and testing sets
# This is CRUCIAL to prevent overfitting and get realistic performance estimates
#
# Training set: Used to train the model (learn patterns)
# Testing set: Used to evaluate the model (unseen data)
#
# test_size=0.2: 80% training, 20% testing (common split)
# random_state=42: Ensures reproducible splits

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print("Data Split for Regression:")
print(f"Training set: {X_train_reg.shape[0]} samples ({X_train_reg.shape[0]/len(X_reg)*100:.1f}%)")
print(f"Testing set: {X_test_reg.shape[0]} samples ({X_test_reg.shape[0]/len(X_reg)*100:.1f}%)")
print(f"\nTraining target range: {y_train_reg.min():.2f} to {y_train_reg.max():.2f} MW")
print(f"Testing target range: {y_test_reg.min():.2f} to {y_test_reg.max():.2f} MW")

In [ ]:
# Scale features for better model performance
# Many ML algorithms perform better with scaled features
# IMPORTANT: Fit scaler on training data only, then transform both train and test

scaler_reg = StandardScaler()

# Fit scaler on training data and transform
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)

# Transform test data using the same scaler (fitted on training data)
# Never fit on test data - this would be data leakage!
X_test_reg_scaled = scaler_reg.transform(X_test_reg)

print("Features scaled using StandardScaler")
print(f"Training set scaled shape: {X_train_reg_scaled.shape}")
print(f"Testing set scaled shape: {X_test_reg_scaled.shape}")

In [ ]:
# Train and compare multiple regression algorithms

# Dictionary to store models and their predictions
regression_models = {}
regression_results = {}

# Model 1: Linear Regression
# Assumes linear relationship between features and target
# Simple, interpretable, fast
print("Training Linear Regression...")
lr_model = LinearRegression()
lr_model.fit(X_train_reg_scaled, y_train_reg)
lr_pred = lr_model.predict(X_test_reg_scaled)
regression_models['Linear Regression'] = lr_model
regression_results['Linear Regression'] = lr_pred

# Model 2: Ridge Regression (L2 regularization)
# Adds penalty for large coefficients to prevent overfitting
# alpha: regularization strength (higher = more regularization)
print("Training Ridge Regression...")
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_reg_scaled, y_train_reg)
ridge_pred = ridge_model.predict(X_test_reg_scaled)
regression_models['Ridge'] = ridge_model
regression_results['Ridge'] = ridge_pred

# Model 3: Random Forest Regressor
# Ensemble of decision trees
# Can capture non-linear relationships
# n_estimators: number of trees in the forest
# max_depth: maximum depth of each tree
print("Training Random Forest Regressor...")
rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train_reg_scaled, y_train_reg)
rf_pred = rf_model.predict(X_test_reg_scaled)
regression_models['Random Forest'] = rf_model
regression_results['Random Forest'] = rf_pred

print("\nAll regression models trained successfully!")

In [ ]:
# Evaluate regression models using multiple metrics

print("="*80)
print("REGRESSION MODEL EVALUATION - LOAD FORECASTING")
print("="*80)

# Create results DataFrame
results_df = pd.DataFrame(columns=['Model', 'RMSE', 'MAE', 'R²', 'MAPE (%)'])

for model_name, predictions in regression_results.items():
    # Root Mean Squared Error (RMSE)
    # Measures average prediction error in same units as target
    # Lower is better
    rmse = np.sqrt(mean_squared_error(y_test_reg, predictions))
    
    # Mean Absolute Error (MAE)
    # Average absolute difference between predicted and actual
    # More robust to outliers than RMSE
    mae = mean_absolute_error(y_test_reg, predictions)
    
    # R² Score (Coefficient of Determination)
    # Proportion of variance explained by the model
    # Range: 0 to 1 (higher is better, 1 = perfect prediction)
    r2 = r2_score(y_test_reg, predictions)
    
    # Mean Absolute Percentage Error (MAPE)
    # Percentage error, useful for comparing across different scales
    mape = np.mean(np.abs((y_test_reg - predictions) / y_test_reg)) * 100
    
    # Add to results
    results_df = pd.concat([results_df, pd.DataFrame({
        'Model': [model_name],
        'RMSE': [rmse],
        'MAE': [mae],
        'R²': [r2],
        'MAPE (%)': [mape]
    })], ignore_index=True)

# Sort by R² score (descending)
results_df = results_df.sort_values('R²', ascending=False).reset_index(drop=True)

print("\n" + results_df.to_string(index=False))
print("\n" + "="*80)

# Identify best model
best_model = results_df.iloc[0]['Model']
best_r2 = results_df.iloc[0]['R²']
print(f"\nBest Model: {best_model} (R² = {best_r2:.4f})")
print("="*80)

In [ ]:
# Visualize regression results
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Regression Model Comparison: Load Forecasting', fontsize=16, fontweight='bold')

# Plot 1: Actual vs Predicted for best model
best_pred = regression_results[best_model]
axes[0, 0].scatter(y_test_reg, best_pred, alpha=0.6, s=30, edgecolor='black', linewidth=0.5)
axes[0, 0].plot([y_test_reg.min(), y_test_reg.max()], 
                [y_test_reg.min(), y_test_reg.max()], 
                'r--', linewidth=2, label='Perfect Prediction')
axes[0, 0].set_xlabel('Actual Load (MW)', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Predicted Load (MW)', fontsize=11, fontweight='bold')
axes[0, 0].set_title(f'{best_model} - Actual vs Predicted', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Prediction errors
errors = y_test_reg - best_pred
axes[0, 1].hist(errors, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0, 1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
axes[0, 1].axvline(x=errors.mean(), color='green', linestyle='--', linewidth=2, 
                   label=f'Mean Error: {errors.mean():.2f} MW')
axes[0, 1].set_xlabel('Prediction Error (MW)', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Prediction Error Distribution', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Model comparison - R² scores
models = results_df['Model'].tolist()
r2_scores = results_df['R²'].tolist()
colors = ['green' if r2 == max(r2_scores) else 'steelblue' for r2 in r2_scores]
axes[1, 0].barh(models, r2_scores, color=colors, edgecolor='black', linewidth=1.5)
axes[1, 0].set_xlabel('R² Score', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Model Comparison: R² Scores', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='x')
for i, (model, r2) in enumerate(zip(models, r2_scores)):
    axes[1, 0].text(r2 + 0.01, i, f'{r2:.4f}', va='center', fontweight='bold')

# Plot 4: Time series of predictions vs actual
# Show first 200 test samples
sample_size = min(200, len(y_test_reg))
sample_indices = range(sample_size)
axes[1, 1].plot(sample_indices, y_test_reg.iloc[:sample_size].values, 
                linewidth=2, label='Actual Load', color='blue', marker='o', markersize=3)
axes[1, 1].plot(sample_indices, best_pred[:sample_size], 
                linewidth=2, label=f'{best_model} Prediction', color='red', 
                linestyle='--', marker='s', markersize=3)
axes[1, 1].set_xlabel('Sample Index', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Load (MW)', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Actual vs Predicted (First 200 Test Samples)', fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Regression visualization complete")

## Section 5: SUPERVISED LEARNING - CLASSIFICATION

### Use Case: Equipment Fault Detection
Classify equipment status as Normal, Voltage Fault, Frequency Fault, or Overload.

In [ ]:
# Prepare data for classification (fault detection)
# Goal: Predict equipment_status from measurements

# Select features (predictors)
feature_cols_classification = ['voltage_kv', 'current_a', 'frequency_hz', 'power_factor',
                              'load_mw', 'temperature_c', 'hour', 'is_weekend']

# Create feature matrix X and target vector y
X_clf = df[feature_cols_classification].copy()
y_clf = df['equipment_status'].copy()

print("Classification Task: Equipment Fault Detection")
print(f"Features (X): {X_clf.shape}")
print(f"Target (y): {y_clf.shape}")
print(f"\nFeature columns:\n{feature_cols_classification}")
print(f"\nTarget classes:")
status_labels = {0: 'Normal', 1: 'Voltage Fault', 2: 'Frequency Fault', 3: 'Overload'}
for status, count in y_clf.value_counts().sort_index().items():
    print(f"  {status}: {status_labels[status]} ({count} samples, {count/len(y_clf)*100:.1f}%)")

In [ ]:
# Split data for classification
# Use stratify to maintain class distribution in train and test sets
# This is important for imbalanced datasets

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

print("Data Split for Classification:")
print(f"Training set: {X_train_clf.shape[0]} samples")
print(f"Testing set: {X_test_clf.shape[0]} samples")

print("\nClass distribution in training set:")
for status, count in y_train_clf.value_counts().sort_index().items():
    print(f"  {status_labels[status]}: {count} ({count/len(y_train_clf)*100:.1f}%)")

print("\nClass distribution in testing set:")
for status, count in y_test_clf.value_counts().sort_index().items():
    print(f"  {status_labels[status]}: {count} ({count/len(y_test_clf)*100:.1f}%)")

In [ ]:
# Scale features for classification
scaler_clf = StandardScaler()
X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf)
X_test_clf_scaled = scaler_clf.transform(X_test_clf)

print("Features scaled for classification")
print(f"Training set scaled shape: {X_train_clf_scaled.shape}")
print(f"Testing set scaled shape: {X_test_clf_scaled.shape}")

In [ ]:
# Train and compare multiple classification algorithms

classification_models = {}
classification_results = {}

# Model 1: Logistic Regression
# Despite the name, it's a classification algorithm
# Uses logistic function to model probability of classes
# max_iter: maximum iterations for convergence
print("Training Logistic Regression...")
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_clf_scaled, y_train_clf)
log_reg_pred = log_reg.predict(X_test_clf_scaled)
classification_models['Logistic Regression'] = log_reg
classification_results['Logistic Regression'] = log_reg_pred

# Model 2: Decision Tree Classifier
# Creates tree of decisions based on feature values
# Interpretable, can handle non-linear relationships
# max_depth: prevents overfitting by limiting tree depth
print("Training Decision Tree...")
dt_clf = DecisionTreeClassifier(max_depth=10, random_state=42)
dt_clf.fit(X_train_clf_scaled, y_train_clf)
dt_pred = dt_clf.predict(X_test_clf_scaled)
classification_models['Decision Tree'] = dt_clf
classification_results['Decision Tree'] = dt_pred

# Model 3: Random Forest Classifier
# Ensemble of decision trees (voting)
# More robust than single decision tree
print("Training Random Forest...")
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_clf.fit(X_train_clf_scaled, y_train_clf)
rf_pred = rf_clf.predict(X_test_clf_scaled)
classification_models['Random Forest'] = rf_clf
classification_results['Random Forest'] = rf_pred

# Model 4: K-Nearest Neighbors
# Classifies based on k nearest training examples
# Simple, non-parametric
# n_neighbors: number of neighbors to consider
print("Training K-Nearest Neighbors...")
knn_clf = KNeighborsClassifier(n_neighbors=5)
knn_clf.fit(X_train_clf_scaled, y_train_clf)
knn_pred = knn_clf.predict(X_test_clf_scaled)
classification_models['KNN'] = knn_clf
classification_results['KNN'] = knn_pred

print("\nAll classification models trained successfully!")

In [ ]:
# Evaluate classification models

print("="*80)
print("CLASSIFICATION MODEL EVALUATION - FAULT DETECTION")
print("="*80)

# Create results DataFrame
clf_results_df = pd.DataFrame(columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score'])

for model_name, predictions in classification_results.items():
    # Accuracy: proportion of correct predictions
    # (TP + TN) / Total
    accuracy = accuracy_score(y_test_clf, predictions)
    
    # Precision: of predicted positives, how many are actually positive
    # TP / (TP + FP)
    # Use weighted average for multi-class
    precision = precision_score(y_test_clf, predictions, average='weighted', zero_division=0)
    
    # Recall (Sensitivity): of actual positives, how many did we predict
    # TP / (TP + FN)
    recall = recall_score(y_test_clf, predictions, average='weighted', zero_division=0)
    
    # F1-Score: harmonic mean of precision and recall
    # 2 × (Precision × Recall) / (Precision + Recall)
    f1 = f1_score(y_test_clf, predictions, average='weighted', zero_division=0)
    
    # Add to results
    clf_results_df = pd.concat([clf_results_df, pd.DataFrame({
        'Model': [model_name],
        'Accuracy': [accuracy],
        'Precision': [precision],
        'Recall': [recall],
        'F1-Score': [f1]
    })], ignore_index=True)

# Sort by F1-Score
clf_results_df = clf_results_df.sort_values('F1-Score', ascending=False).reset_index(drop=True)

print("\n" + clf_results_df.to_string(index=False))
print("\n" + "="*80)

# Identify best model
best_clf_model = clf_results_df.iloc[0]['Model']
best_f1 = clf_results_df.iloc[0]['F1-Score']
print(f"\nBest Model: {best_clf_model} (F1-Score = {best_f1:.4f})")
print("="*80)

In [ ]:
# Create confusion matrix for best classification model
best_clf_pred = classification_results[best_clf_model]

# Confusion matrix shows actual vs predicted classifications
# Rows: actual classes, Columns: predicted classes
# Diagonal: correct predictions
cm = confusion_matrix(y_test_clf, best_clf_pred)

# Visualize confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=list(status_labels.values()),
            yticklabels=list(status_labels.values()),
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Status', fontsize=12, fontweight='bold')
plt.ylabel('Actual Status', fontsize=12, fontweight='bold')
plt.title(f'Confusion Matrix: {best_clf_model}', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Print detailed classification report
print("\nDetailed Classification Report:")
print("="*80)
print(classification_report(y_test_clf, best_clf_pred, 
                          target_names=list(status_labels.values()),
                          zero_division=0))

## Section 6: UNSUPERVISED LEARNING - CLUSTERING

### Use Case: Load Pattern Segmentation
Group time periods with similar load characteristics (no labels needed).

In [ ]:
# Prepare data for clustering
# We'll cluster based on load patterns
# No target variable needed - this is unsupervised!

# Select features for clustering
feature_cols_clustering = ['load_mw', 'temperature_c', 'hour', 'day_of_week',
                          'hour_sin', 'hour_cos']

X_cluster = df[feature_cols_clustering].copy()

# Scale features (important for clustering)
scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)

print("Clustering Task: Load Pattern Segmentation")
print(f"Features: {X_cluster.shape}")
print(f"Feature columns: {feature_cols_clustering}")
print("\nNo target variable - unsupervised learning!")

In [ ]:
# Apply K-Means Clustering
# K-Means groups data into k clusters
# Each cluster has a centroid (center point)
# Points are assigned to nearest centroid

# Try different numbers of clusters
n_clusters_options = [2, 3, 4, 5]

# Store results
clustering_results = {}

for n_clusters in n_clusters_options:
    print(f"\nTrying K-Means with {n_clusters} clusters...")
    
    # Create and fit K-Means model
    # n_init: number of times to run with different initializations
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_cluster_scaled)
    
    # Calculate silhouette score (quality metric)
    # Range: -1 to 1 (higher is better)
    # Measures how similar points are within cluster vs other clusters
    silhouette = silhouette_score(X_cluster_scaled, cluster_labels)
    
    # Calculate Davies-Bouldin index (another quality metric)
    # Lower is better
    # Measures average similarity between clusters
    db_index = davies_bouldin_score(X_cluster_scaled, cluster_labels)
    
    clustering_results[n_clusters] = {
        'model': kmeans,
        'labels': cluster_labels,
        'silhouette': silhouette,
        'db_index': db_index
    }
    
    print(f"  Silhouette Score: {silhouette:.4f}")
    print(f"  Davies-Bouldin Index: {db_index:.4f}")
    
    # Show cluster sizes
    unique, counts = np.unique(cluster_labels, return_counts=True)
    print(f"  Cluster sizes: {dict(zip(unique, counts))}")

# Find best number of clusters (highest silhouette score)
best_n_clusters = max(clustering_results.keys(), 
                      key=lambda k: clustering_results[k]['silhouette'])
print(f"\nBest number of clusters: {best_n_clusters} (Silhouette: {clustering_results[best_n_clusters]['silhouette']:.4f})")

In [ ]:
# Visualize clustering results
best_labels = clustering_results[best_n_clusters]['labels']
best_model = clustering_results[best_n_clusters]['model']

# Add cluster labels to dataframe
df_clustered = df.copy()
df_clustered['cluster'] = best_labels

# Analyze clusters
print("\nCluster Characteristics:")
print("="*80)
for cluster_id in range(best_n_clusters):
    cluster_data = df_clustered[df_clustered['cluster'] == cluster_id]
    print(f"\nCluster {cluster_id}: {len(cluster_data)} samples ({len(cluster_data)/len(df)*100:.1f}%)")
    print(f"  Average Load: {cluster_data['load_mw'].mean():.2f} MW")
    print(f"  Average Temperature: {cluster_data['temperature_c'].mean():.2f} °C")
    print(f"  Most common hour: {cluster_data['hour'].mode().values[0]}:00")
    print(f"  Weekend samples: {cluster_data['is_weekend'].sum()} ({cluster_data['is_weekend'].sum()/len(cluster_data)*100:.1f}%)")

In [ ]:
# Visualize clusters
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f'K-Means Clustering: {best_n_clusters} Load Pattern Groups', 
             fontsize=16, fontweight='bold')

# Plot 1: Load vs Temperature colored by cluster
scatter1 = axes[0, 0].scatter(df_clustered['temperature_c'], df_clustered['load_mw'],
                             c=df_clustered['cluster'], cmap='viridis', 
                             alpha=0.6, s=30, edgecolor='black', linewidth=0.5)
axes[0, 0].set_xlabel('Temperature (°C)', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Load (MW)', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Load vs Temperature by Cluster', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)
plt.colorbar(scatter1, ax=axes[0, 0], label='Cluster')

# Plot 2: Load by hour colored by cluster
for cluster_id in range(best_n_clusters):
    cluster_data = df_clustered[df_clustered['cluster'] == cluster_id]
    hourly_avg = cluster_data.groupby('hour')['load_mw'].mean()
    axes[0, 1].plot(hourly_avg.index, hourly_avg.values, 
                   linewidth=2, marker='o', label=f'Cluster {cluster_id}')
axes[0, 1].set_xlabel('Hour of Day', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Average Load (MW)', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Load Profile by Cluster', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Cluster sizes
cluster_sizes = df_clustered['cluster'].value_counts().sort_index()
axes[1, 0].bar(cluster_sizes.index, cluster_sizes.values, 
              color='steelblue', edgecolor='black', linewidth=1.5)
axes[1, 0].set_xlabel('Cluster', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Number of Samples', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Cluster Size Distribution', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(cluster_sizes.values):
    axes[1, 0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Plot 4: Silhouette scores for different k values
k_values = list(clustering_results.keys())
silhouette_scores = [clustering_results[k]['silhouette'] for k in k_values]
axes[1, 1].plot(k_values, silhouette_scores, linewidth=3, marker='o', 
               markersize=10, color='green')
axes[1, 1].scatter([best_n_clusters], [clustering_results[best_n_clusters]['silhouette']],
                  s=300, color='red', zorder=5, edgecolor='black', linewidth=2,
                  label=f'Best k={best_n_clusters}')
axes[1, 1].set_xlabel('Number of Clusters (k)', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Silhouette Score', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Optimal Number of Clusters', fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_xticks(k_values)

plt.tight_layout()
plt.show()

print("Clustering visualization complete")

## Section 7: Cross-Validation

More robust evaluation method than single train-test split.

In [ ]:
# Demonstrate cross-validation for regression
# Cross-validation splits data into k folds
# Trains on k-1 folds, tests on remaining fold
# Repeats k times, each fold used once for testing
# Provides more robust performance estimate

print("Cross-Validation for Regression Models")
print("="*80)

# Use 5-fold cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Test best regression model
print(f"\nModel: {best_model}")
print(f"Cross-validation: 5-fold")

# Get cross-validation scores
# scoring='r2': use R² as metric
# cv=cv: use our KFold object
cv_scores = cross_val_score(regression_models[best_model], 
                           X_reg, y_reg, 
                           cv=cv, scoring='r2')

print(f"\nR² scores for each fold:")
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {score:.4f}")

print(f"\nMean R²: {cv_scores.mean():.4f}")
print(f"Std Dev: {cv_scores.std():.4f}")
print(f"95% Confidence Interval: {cv_scores.mean():.4f} ± {1.96 * cv_scores.std():.4f}")

print("\n" + "="*80)
print("Cross-validation provides more reliable performance estimate")
print("Lower std dev indicates model is consistent across different data splits")
print("="*80)

## Section 8: Summary and Best Practices

In [ ]:
# Create comprehensive summary
print("="*80)
print("MACHINE LEARNING SUMMARY")
print("="*80)

print("\n1. SUPERVISED LEARNING - REGRESSION (Load Forecasting)")
print("-" * 80)
print(f"Task: Predict continuous load values")
print(f"Best Model: {best_model}")
print(f"Performance: R² = {best_r2:.4f}")
print(f"RMSE: {results_df[results_df['Model'] == best_model]['RMSE'].values[0]:.2f} MW")
print(f"Use Cases: Load forecasting, price prediction, energy consumption estimation")

print("\n2. SUPERVISED LEARNING - CLASSIFICATION (Fault Detection)")
print("-" * 80)
print(f"Task: Classify equipment status into 4 categories")
print(f"Best Model: {best_clf_model}")
print(f"Performance: F1-Score = {best_f1:.4f}")
print(f"Accuracy: {clf_results_df[clf_results_df['Model'] == best_clf_model]['Accuracy'].values[0]:.4f}")
print(f"Use Cases: Fault detection, equipment health classification, alarm prioritization")

print("\n3. UNSUPERVISED LEARNING - CLUSTERING (Pattern Discovery)")
print("-" * 80)
print(f"Task: Group similar load patterns without labels")
print(f"Best Configuration: {best_n_clusters} clusters")
print(f"Performance: Silhouette Score = {clustering_results[best_n_clusters]['silhouette']:.4f}")
print(f"Use Cases: Customer segmentation, load pattern analysis, anomaly detection")

print("\n" + "="*80)
print("KEY TAKEAWAYS")
print("="*80)
print("✓ Always split data into train/test sets")
print("✓ Scale features for most algorithms")
print("✓ Use cross-validation for robust evaluation")
print("✓ Choose metrics appropriate for your problem")
print("✓ Compare multiple algorithms before selecting one")
print("✓ Understand the trade-offs of each algorithm")
print("="*80)

## What This Means for Electrical Engineers

### Industry Relevance:

1. **Load Forecasting (Regression)**: Utilities use ML regression models to forecast load 1-hour to 1-year ahead. Accurate forecasts save millions through:
   - Optimal unit commitment (which generators to run)
   - Energy market bidding strategies
   - Reserve requirement planning

2. **Fault Detection (Classification)**: ML classifiers identify equipment faults before catastrophic failures:
   - Transformer diagnostics (dissolved gas analysis)
   - Circuit breaker health assessment
   - Transmission line fault classification
   - Reduces unplanned outages by 30-50%

3. **Pattern Discovery (Clustering)**: Unsupervised learning reveals hidden patterns:
   - Customer load profile segmentation for tariff design
   - Identifying similar equipment for maintenance scheduling
   - Detecting unusual consumption patterns (theft, malfunction)

4. **Real-Time Applications**: Modern SCADA systems integrate ML for:
   - Dynamic security assessment
   - Predictive maintenance
   - Renewable generation forecasting
   - Demand response optimization

### Key Takeaways:

- **Supervised learning needs labeled data**: You must have input-output pairs
- **Unsupervised learning finds patterns**: No labels needed, discovers structure
- **Train-test split is mandatory**: Never evaluate on training data (overfitting)
- **Cross-validation is more robust**: Better than single train-test split
- **Feature scaling matters**: Most algorithms perform better with scaled features
- **No single best algorithm**: Try multiple approaches and compare
- **Metrics must match goals**: Use appropriate metrics for your problem
- **Simple models often work well**: Don't always need complex algorithms

### Common Mistakes:

- **Not splitting data**: Training and testing on same data gives falsely optimistic results
- **Fitting scaler on test data**: This is data leakage - always fit on training only
- **Ignoring class imbalance**: Classification with imbalanced classes needs special handling
- **Wrong metrics**: Using accuracy for imbalanced classification is misleading
- **Overfitting**: Model performs well on training but poorly on test data
- **Not validating assumptions**: Some algorithms assume normally distributed features

### Pro Tips:

- Start with simple models (Linear Regression, Logistic Regression)
- Visualize your data before modeling
- Use domain knowledge to engineer features
- Monitor both training and test performance
- Save your trained models for deployment
- Document your preprocessing steps
- Consider computational requirements for production
- Retrain models periodically as data evolves

### Algorithm Selection Guide:

**Regression (Predict continuous values):**
- Linear Regression: Fast, interpretable, assumes linearity
- Ridge/Lasso: Like linear regression but prevents overfitting
- Random Forest: Handles non-linearity, robust, less interpretable

**Classification (Predict categories):**
- Logistic Regression: Fast, interpretable, probabilistic outputs
- Decision Trees: Interpretable, handles non-linearity, prone to overfitting
- Random Forest: Robust, accurate, less interpretable
- SVM: Good for high-dimensional data, complex decision boundaries
- KNN: Simple, no training phase, slow for large datasets

**Clustering (Group similar data):**
- K-Means: Fast, requires specifying k, assumes spherical clusters
- DBSCAN: Finds arbitrary shapes, determines k automatically, sensitive to parameters
- Hierarchical: Creates dendrogram, interpretable, computationally expensive

### Next Steps:

In the next notebook (Model Evaluation), we'll dive deeper into:
- Advanced evaluation metrics
- Confusion matrices and ROC curves
- Hyperparameter tuning
- Model selection strategies
- Bias-variance tradeoff

Understanding these ML fundamentals is essential before moving to specific algorithms in subsequent modules.